# Phase 1 — Markov Chain Foundations

Companion notebook to `notes/phase1-markov-foundations.md`. Reproduces the worked examples and the headcount classification numerically using `src.dtmc` and `src.simulation`.

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd

from src.dtmc import MarkovChain
from src.simulation import simulate_paths, state_distribution_at

np.set_printoptions(precision=4, suppress=True)

## 1. Three-state worked example

Compute $P^2$ and $P^3$ to match the values derived by hand.

In [ ]:
P3 = np.array(
    [
        [0.7, 0.3, 0.0],
        [0.0, 0.8, 0.2],
        [0.0, 0.0, 1.0],
    ]
)
mc3 = MarkovChain(P3, state_labels=["Junior", "Mid", "Senior"])

print("P^1 =\n", mc3.n_step(1))
print("\nP^2 =\n", mc3.n_step(2))
print("\nP^3 =\n", mc3.n_step(3))

## 2. Headcount chain — recycling vs absorbing variants

The recycling variant ($p_{41} = 0.5$) is irreducible. The absorbing variant ($p_{44} = 1$) splits into singleton communication classes.

In [ ]:
labels = ["Junior", "Mid", "Senior", "Exit"]

P_recycle = np.array(
    [
        [0.93, 0.03, 0.00, 0.04],
        [0.00, 0.96, 0.02, 0.02],
        [0.00, 0.00, 0.99, 0.01],
        [0.50, 0.00, 0.00, 0.50],
    ]
)
P_absorb = P_recycle.copy()
P_absorb[3] = [0.0, 0.0, 0.0, 1.0]

mc_recycle = MarkovChain(P_recycle, state_labels=labels)
mc_absorb = MarkovChain(P_absorb, state_labels=labels)

print("recycling:  irreducible?", mc_recycle.is_irreducible(),
      " absorbing chain?", mc_recycle.is_absorbing())
print("absorbing:  irreducible?", mc_absorb.is_irreducible(),
      " absorbing chain?", mc_absorb.is_absorbing())

print("\ncommunication classes (recycling):", mc_recycle.communication_classes())
print("communication classes (absorbing):", mc_absorb.communication_classes())

## 3. Distribution evolution

Start with all Juniors and evolve forward 1, 6, and 12 months under the recycling variant.

In [ ]:
pi0 = np.array([1.0, 0.0, 0.0, 0.0])
rows = {n: mc_recycle.evolve(pi0, n) for n in (0, 1, 3, 6, 12, 24)}
df = pd.DataFrame(rows, index=labels).T
df.index.name = "month"
df.round(4)

## 4. Empirical → analytical convergence (sanity check)

Simulate 5{,}000 trajectories of length 12 and compare the empirical distribution at month 12 with $P^{12} \cdot e_J$.

In [ ]:
paths = simulate_paths(mc_recycle, initial=0, n_steps=12, n_paths=5000, seed=2026)
empirical = state_distribution_at(paths, step=12, n_states=mc_recycle.n_states)
analytical = mc_recycle.evolve(pi0, 12)

comparison = pd.DataFrame(
    {"empirical": empirical, "P^12 · e_J": analytical}, index=labels
).round(4)
comparison